# Part 2: Retrieval-Augmented Generation (RAG)

Team 3 CozyDay | Lin

Uses Rishabh's frozen slm_exp5.pt checkpoint.

## Reproducible setup and checkpoint access

Obtain the original `slm_exp5.pt` from the team's private checkpoint share; do not retrain it or commit weights. Launch this notebook from the repository root or `a7_part2`.

Before starting Jupyter/PyCharm, set the environment variable `SLM_CHECKPOINT_PATH` to your local checkpoint file. The notebook reads this variable without overriding it. Alternatively, place the private checkpoint in a relative location searched by the loading cell, such as `model_checkpoints/slm_exp5.pt` relative to the repository root. Keep that directory outside staged changes.

Example in PowerShell before launching the notebook application:

```powershell
$env:SLM_CHECKPOINT_PATH = (Resolve-Path "../model_checkpoints/slm_exp5.pt").Path
```

Use the path appropriate to your own working directory. If the notebook application was already running, restart it so its kernel inherits the variable.

Expected checkpoint SHA-256: `408c27aa60346f00a41b1cef451e434c277493fc8329061ddaec060043e78a1f`.

Generation settings: seed 42 reset before each response, temperature 1.0, 30 new characters, and a sliding 32-character input window. The original baseline ends with the question alone; RAG and the supplemental matched baseline end with a trailing space. Keep the original 25 records and the five supplemental records separate.

Verified reviewer environment: Python 3.12.14, PyTorch 2.14.1+cpu, pandas 2.2.3, CPU. Exact sampled text is not guaranteed across CPU/GPU backends or software versions. Before fine-tuning, reproduce the chosen baseline and compare all five responses. The instructor requirements were checked against the supplied assignment handout. Provenance against the unavailable Week 6 retrieval notebook remains unverified.


### Revision following Ashley's PR review

`part2_matched_baseline.csv` is the authoritative no-RAG baseline for Part 3 and Part 4. Shivani must reproduce its five responses with the same checkpoint, question plus trailing space, seed 42, temperature 1.0, and 30 generated characters before fine-tuning. The original question-only baseline is historical and must not be substituted for it.

`part2_rag_results.csv` retains the original 25 records unchanged. The corrected 20 RAG runs are exported to `part2_rag_results_revised.csv`. `part2_quality_evaluation.csv` contains the new three-criterion evaluation of the authoritative baseline and corrected RAG runs. Its revised scores must not be confused with the historical proxy scores in the original CSVs.

The correction reverses prompt document order, not retrieval rank: lower-ranked documents appear first and the highest-ranked document immediately precedes the unchanged question. All full prompts and actual initial contexts are recorded. With this 32-character model, all K values for a given question have the same surviving context; the answer word survives, but the subject in the fact is still truncated. These are four requested retrieval configurations, not four different amounts of evidence visible to the generator. We do not change the questions or model to conceal that limitation.

## 1. Load the Frozen Part 1 Checkpoint

In [1]:

import torch
import torch.nn as nn
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import os

candidates = [
    os.environ.get("SLM_CHECKPOINT_PATH"),
    "../../model_checkpoints/slm_exp5.pt",
    "../model_checkpoints/slm_exp5.pt",
    "model_checkpoints/slm_exp5.pt",
    "../slm_exp5.pt",
    "slm_exp5.pt",
]

CHECKPOINT_PATH = next(
    (
        Path(p).expanduser().resolve()
        for p in candidates
        if p and Path(p).expanduser().is_file()
    ),
    None
)

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Checkpoint not found. Set SLM_CHECKPOINT_PATH "
        "to the location of slm_exp5.pt."
    )

print("Checkpoint:", CHECKPOINT_PATH)

saved = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

chars = saved["vocab"]
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
config = saved["config"]

print("Device:", DEVICE)
print("Vocabulary:", repr("".join(chars)))
print("Architecture:", config)


Checkpoint: D:\INFO490\model_checkpoints\slm_exp5.pt
Device: cpu
Vocabulary: '\n .abcdefghijklmnoprstuvwxy'
Architecture: {'embedding_dim': 96, 'num_heads': 4, 'num_layers': 2, 'feedforward_dim': 384, 'context_length': 32}


## 2. Reuse the Original Transformer Architecture

In [2]:
# ============================================================
# CELL 9A: BUILD A TRANSFORMER BLOCK
# ============================================================
# Purpose:
# A Transformer block combines:
#
# 1. Multi-head self-attention
# 2. Residual connections
# 3. Layer normalization
# 4. Feed-forward neural network
#
# Multi-head attention allows the model to learn different
# attention patterns simultaneously.
#
# Residual connections help information and gradients flow
# through the network.
#
# Layer normalization stabilizes representations.
#
# The feed-forward network transforms each token's features.
#
# PyTorch provides MultiheadAttention, which implements
# the Q/K/V projections and attention computation.
#
# We use batch_first=True:
# (batch, sequence, embedding_dimension)
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()

        # Main Component 1
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=0.0,
            batch_first=True)

        # Supporting Component 1
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        # Main Component 2
        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, feedforward_dim),
            nn.ReLU(),
            nn.Linear(feedforward_dim, embedding_dim))


    def forward(self, x):

        T = x.shape[1]


        # True means a position is blocked from attention.
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)

        # Supporting Component 2
        # Attention + residual connection.
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)

        # Residual 1: Original representation + attention output
        x = x + attended

        # Residual 2: Updated representation + feed-forward output
        # Feed-forward network + residual connection.
        x = x + self.feedforward(self.ln2(x))

        return x

# ============================================================
# CELL 9B: BUILDING OUR COMPLETE TINY TRANSFORMER
# ============================================================
# Purpose:
# Combine all the components into one language model.
#
# Architecture:
#
# Token IDs
#    ↓
# Token Embeddings + Position Embeddings
#    ↓
# Transformer Block(s)
#    ↓
# Layer Normalization
#    ↓
# Linear Output Layer
#    ↓
# Logits for every vocabulary token
#
# The output tensor has shape:
#
# (batch_size, sequence_length, vocabulary_size)
#
# Each position produces scores for the NEXT character.
# ============================================================

class TinyTransformerLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, feedforward_dim)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):

        B, T = input_ids.shape

        if T > self.context_length:
            raise ValueError("Input sequence exceeds context length.")

        positions = torch.arange(T, device=input_ids.device)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)

        return self.output(x)

## 3. Restore Trained Weights

In [3]:

model = TinyTransformerLM(
    vocab_size=len(chars),
    embedding_dim=config["embedding_dim"],
    num_heads=config["num_heads"],
    num_layers=config["num_layers"],
    feedforward_dim=config["feedforward_dim"],
    context_length=config["context_length"]
).to(DEVICE)

model.load_state_dict(saved["model_state"], strict=True)
model.eval()

parameter_count = sum(p.numel() for p in model.parameters())
assert parameter_count == 232155
assert model.context_length == 32

print("Checkpoint loaded successfully!")
print("Parameters:", parameter_count)
print("Context length:", model.context_length)


Checkpoint loaded successfully!
Parameters: 232155
Context length: 32


In [4]:
# Part 2: Knowledge Base

knowledge_base = [
    {"id": "D1", "text": "mia likes apples"},
    {"id": "D2", "text": "sam likes music"},
    {"id": "D3", "text": "tom likes swimming"},
    {"id": "D4", "text": "anna likes painting"},
    {"id": "D5", "text": "lily likes reading"},
]

questions = [
    {"id": "Q1", "question": "what does mia like", "answer": "apples", "doc_id": "D1"},
    {"id": "Q2", "question": "what does sam like", "answer": "music", "doc_id": "D2"},
    {"id": "Q3", "question": "what does tom like", "answer": "swimming", "doc_id": "D3"},
    {"id": "Q4", "question": "what does anna like", "answer": "painting", "doc_id": "D4"},
    {"id": "Q5", "question": "what does lily like", "answer": "reading", "doc_id": "D5"},
]

allowed = set(chars)

for doc in knowledge_base:
    assert set(doc["text"]).issubset(allowed)

for item in questions:
    assert set(item["question"]).issubset(allowed)
    assert set(item["answer"]).issubset(allowed)

print("Knowledge base validated!")
print("Documents:", len(knowledge_base))
print("Questions:", len(questions))

Knowledge base validated!
Documents: 5
Questions: 5


In [5]:
import re

class KnowledgeRetriever:
    def __init__(self, documents):
        self.documents = documents

    def tokenize(self, text):
        return set(re.findall(r"[a-z]+", text.lower()))

    def retrieve(self, query, top_k=1):
        query_words = self.tokenize(query)

        results = []

        for doc in self.documents:
            doc_words = self.tokenize(doc["text"])

            # Word overlap similarity
            score = len(query_words & doc_words)

            results.append({
                "id": doc["id"],
                "text": doc["text"],
                "score": score
            })

        results.sort(
            key=lambda item: item["score"],
            reverse=True
        )

        return results[:top_k]


retriever = KnowledgeRetriever(knowledge_base)

# Verify each question retrieves the correct document first.
for item in questions:
    retrieved = retriever.retrieve(item["question"], top_k=5)

    top_id = retrieved[0]["id"]

    print(
        item["id"],
        "| Expected:", item["doc_id"],
        "| Retrieved:", top_id,
        "| Ranking:", [doc["id"] for doc in retrieved]
    )

    assert top_id == item["doc_id"], (
        f"Retrieval failed for {item['id']}"
    )

print("\nAll five retrieval checks passed!")

Q1 | Expected: D1 | Retrieved: D1 | Ranking: ['D1', 'D2', 'D3', 'D4', 'D5']
Q2 | Expected: D2 | Retrieved: D2 | Ranking: ['D2', 'D1', 'D3', 'D4', 'D5']
Q3 | Expected: D3 | Retrieved: D3 | Ranking: ['D3', 'D1', 'D2', 'D4', 'D5']
Q4 | Expected: D4 | Retrieved: D4 | Ranking: ['D4', 'D1', 'D2', 'D3', 'D5']
Q5 | Expected: D5 | Retrieved: D5 | Ranking: ['D5', 'D1', 'D2', 'D3', 'D4']

All five retrieval checks passed!


In [6]:
# Part 2: No-RAG Baseline
import torch
import pandas as pd

SEED = 42
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 30

@torch.no_grad()
def rag_generate(prompt, seed=SEED):
    # Check that every input character belongs to the model vocabulary.
    unknown = sorted(set(prompt) - set(stoi))
    if unknown:
        raise ValueError(f"Unsupported characters: {unknown}")

    # Fix seed BEFORE each generation for reproducibility.
    torch.manual_seed(seed)

    model.eval()
    context = prompt[-model.context_length:]
    tokens = torch.tensor(
        [[stoi[ch] for ch in context]],
        dtype=torch.long,
        device=DEVICE
    )

    generated = []

    for _ in range(MAX_NEW_TOKENS):
        logits = model(tokens)[:, -1, :]
        probabilities = torch.softmax(
            logits / TEMPERATURE, dim=-1
        )

        next_token = torch.multinomial(
            probabilities, num_samples=1
        )

        char = itos[next_token.item()]
        generated.append(char)

        tokens = torch.cat(
            [tokens, next_token], dim=1
        )[:, -model.context_length:]

    return "".join(generated), context


baseline_results = []

for item in questions:
    response, actual_context = rag_generate(
        item["question"],
        seed=SEED
    )

    baseline_results.append({
        "question_id": item["id"],
        "question": item["question"],
        "expected_answer": item["answer"],
        "actual_context": actual_context,
        "response": response
    })

baseline_df = pd.DataFrame(baseline_results)

display(baseline_df)

,question_id,question,expected_answer,actual_context,response
0,Q1,what does mia like,apples,what does mia like,n.\ntransformers use attention
1,Q2,what does sam like,music,what does sam like,n.\ntransformers use attention
2,Q3,what does tom like,swimming,what does tom like,n.\ntransformers use attention
3,Q4,what does anna like,painting,what does anna like,n.\ntransformers use attention
4,Q5,what does lily like,reading,what does lily like,slanguage models can run on lo


In [7]:
# Corrected RAG experiment: preserve the highest-ranked evidence at the suffix.
TOP_K_VALUES = [1, 2, 3, 5]
rag_results = []
for item in questions:
    for k in TOP_K_VALUES:
        retrieved = retriever.retrieve(item["question"], top_k=k)
        prompt_documents = list(reversed(retrieved))
        knowledge_text = "\n".join(doc["text"] for doc in prompt_documents)
        prompt = knowledge_text + "\n" + item["question"] + " "
        response, actual_context = rag_generate(prompt, seed=SEED)
        assert actual_context == prompt[-model.context_length:]
        assert item["answer"] in actual_context
        rag_results.append({
            "experiment_version": "rank_near_question_v2",
            "question_id": item["id"], "question": item["question"],
            "expected_answer": item["answer"], "top_k": k,
            "retrieved_ids": ",".join(d["id"] for d in retrieved),
            "prompt_document_ids": ",".join(d["id"] for d in prompt_documents),
            "full_prompt": prompt, "full_prompt_length": len(prompt),
            "actual_context": actual_context, "context_length": len(actual_context),
            "top_document_fully_visible": retrieved[0]["text"] in actual_context,
            "response": response, "seed": SEED, "temperature": TEMPERATURE,
            "max_new_tokens": MAX_NEW_TOKENS,
        })
rag_df = pd.DataFrame(rag_results)
assert len(rag_df) == 20
assert not rag_df.duplicated(["question_id", "top_k"]).any()
assert rag_df["context_length"].eq(32).all()
display(rag_df)

,experiment_version,question_id,question,expected_answer,top_k,retrieved_ids,prompt_document_ids,full_prompt,full_prompt_length,actual_context,context_length,top_document_fully_visible,response,seed,temperature,max_new_tokens
0,rank_near_question_v2,Q1,what does mia like,apples,1,D1,D1,mia likes apples\nwhat does mia like,36,likes apples\nwhat does mia like,32,False,intelligence.\nstudents create,42,1.0,30
1,rank_near_question_v2,Q1,what does mia like,apples,2,"D1,D2","D2,D1",sam likes music\nmia likes apples\nwhat does m...,52,likes apples\nwhat does mia like,32,False,intelligence.\nstudents create,42,1.0,30
2,rank_near_question_v2,Q1,what does mia like,apples,3,"D1,D2,D3","D3,D2,D1",tom likes swimming\nsam likes music\nmia likes...,71,likes apples\nwhat does mia like,32,False,intelligence.\nstudents create,42,1.0,30
3,rank_near_question_v2,Q1,what does mia like,apples,5,"D1,D2,D3,D4,D5","D5,D4,D3,D2,D1",lily likes reading\nanna likes painting\ntom l...,110,likes apples\nwhat does mia like,32,False,intelligence.\nstudents create,42,1.0,30
4,rank_near_question_v2,Q2,what does sam like,music,1,D2,D2,sam likes music\nwhat does sam like,35,likes music\nwhat does sam like,32,False,s useful for machine learning,42,1.0,30
5,rank_near_question_v2,Q2,what does sam like,music,2,"D2,D1","D1,D2",mia likes apples\nsam likes music\nwhat does s...,52,likes music\nwhat does sam like,32,False,s useful for machine learning,42,1.0,30
6,rank_near_question_v2,Q2,what does sam like,music,3,"D2,D1,D3","D3,D1,D2",tom likes swimming\nmia likes apples\nsam like...,71,likes music\nwhat does sam like,32,False,s useful for machine learning,42,1.0,30
7,rank_near_question_v2,Q2,what does sam like,music,5,"D2,D1,D3,D4,D5","D5,D4,D3,D1,D2",lily likes reading\nanna likes painting\ntom l...,110,likes music\nwhat does sam like,32,False,s useful for machine learning,42,1.0,30
8,rank_near_question_v2,Q3,what does tom like,swimming,1,D3,D3,tom likes swimming\nwhat does tom like,38,kes swimming\nwhat does tom like,32,False,s can run on local computers.\n,42,1.0,30
9,rank_near_question_v2,Q3,what does tom like,swimming,2,"D3,D1","D1,D3",mia likes apples\ntom likes swimming\nwhat doe...,55,kes swimming\nwhat does tom like,32,False,s can run on local computers.\n,42,1.0,30


In [8]:
# Check answer correctness and context visibility

def contains_answer(text, expected):
    return expected.lower() in text.lower()

rag_df["answer_found"] = rag_df.apply(
    lambda row: contains_answer(
        row["response"],
        row["expected_answer"]
    ),
    axis=1
)

rag_df["answer_visible_in_context"] = rag_df.apply(
    lambda row: contains_answer(
        row["actual_context"],
        row["expected_answer"]
    ),
    axis=1
)

print("Correct answers generated:", rag_df["answer_found"].sum(), "/ 20")
print(
    "Expected answer visible in context:",
    rag_df["answer_visible_in_context"].sum(),
    "/ 20"
)

display(
    rag_df[
        [
            "question_id",
            "top_k",
            "retrieved_ids",
            "answer_visible_in_context",
            "answer_found",
            "response"
        ]
    ]
)

Correct answers generated: 0 / 20
Expected answer visible in context: 20 / 20


,question_id,top_k,retrieved_ids,answer_visible_in_context,answer_found,response
0,Q1,1,D1,True,False,intelligence.\nstudents create
1,Q1,2,"D1,D2",True,False,intelligence.\nstudents create
2,Q1,3,"D1,D2,D3",True,False,intelligence.\nstudents create
3,Q1,5,"D1,D2,D3,D4,D5",True,False,intelligence.\nstudents create
4,Q2,1,D2,True,False,s useful for machine learning
5,Q2,2,"D2,D1",True,False,s useful for machine learning
6,Q2,3,"D2,D1,D3",True,False,s useful for machine learning
7,Q2,5,"D2,D1,D3,D4,D5",True,False,s useful for machine learning
8,Q3,1,D3,True,False,s can run on local computers.\n
9,Q3,2,"D3,D1",True,False,s can run on local computers.\n


In [9]:
# Part 2: Quality Scoring

def quality_score(response, expected_answer):
    """
    Reproducible rule-based scoring on a 1-10 scale.
    These are proxy quality scores, not human ratings.
    """
    response = response.lower().strip()
    expected = expected_answer.lower().strip()

    # Correct answer appears as a complete word
    import re
    if re.search(r"\b" + re.escape(expected) + r"\b", response):
        if response == expected:
            return 10
        return 8

    # Partial answer fragment (at least 4 characters)
    if len(expected) >= 4 and expected[:4] in response:
        return 4

    # Text is generated but does not answer the question
    if response and len(response.split()) >= 3:
        return 2

    # Incoherent, incomplete, or nearly empty response
    return 1



# This helper is retained only for historical CSV compatibility.
# The revised three-criterion evaluation below supplies current RAG scores.

def evaluate_response(response, expected_answer, question, visible_context=None):
    """Conservative task-specific rule proxies; not human ratings or entailment."""
    clean = " ".join(response.lower().split()).strip(" .")
    subject = question.split()[2]
    expected = expected_answer.lower()
    accepted = {expected, f"{subject} likes {expected}"}
    # Credit only a complete accepted atomic answer, rather than any substring.
    accuracy = 3 if clean in accepted else 0
    subject_present = re.search(r"\b" + re.escape(subject) + r"\b", clean) is not None
    relevance = 3 if accuracy else (1 if subject_present else 0)
    if visible_context is None:
        support = None
        support_note = "Not applicable: no retrieved information supplied."
    else:
        evidence_present = re.search(r"\b" + re.escape(expected) + r"\b", visible_context) is not None
        support = 3 if accuracy and evidence_present else 0
        support_note = ("Accepted atomic answer matches the answer span in visible evidence."
                        if support else "No correct task answer grounded in visible retrieved evidence.")
    assessed = [accuracy, relevance] + ([] if support is None else [support])
    score = round(1 + 9 * sum(assessed) / (3 * len(assessed)), 2)
    return {
        "factual_accuracy_0_3": accuracy, "relevance_0_3": relevance,
        "retrieved_support_0_3": support, "quality_score": score,
        "accuracy_reason": ("Matches the reference atomic fact." if accuracy else
                            "Does not provide a complete accepted answer to the question."),
        "relevance_reason": ("Answers the preference question." if accuracy else
                             "Mentions the subject but does not answer." if subject_present else
                             "Off-topic or fragmented continuation; does not address the subject preference."),
        "support_reason": support_note,
    }

evaluations = pd.DataFrame([
    evaluate_response(row.response, row.expected_answer, row.question, row.actual_context)
    for row in rag_df.itertuples()
])
rag_df = pd.concat([rag_df.reset_index(drop=True), evaluations], axis=1)
display(rag_df[["question_id", "top_k", "factual_accuracy_0_3", "relevance_0_3",
               "retrieved_support_0_3", "quality_score"]])

,question_id,top_k,factual_accuracy_0_3,relevance_0_3,retrieved_support_0_3,quality_score
0,Q1,1,0,0,0,1.0
1,Q1,2,0,0,0,1.0
2,Q1,3,0,0,0,1.0
3,Q1,5,0,0,0,1.0
4,Q2,1,0,0,0,1.0
5,Q2,2,0,0,0,1.0
6,Q2,3,0,0,0,1.0
7,Q2,5,0,0,0,1.0
8,Q3,1,0,0,0,1.0
9,Q3,2,0,0,0,1.0


In [10]:
# Export corrected runs separately. Never overwrite the original 25 records.
cwd = Path.cwd()
if (cwd / "a7_part2" / "Part2_RAG.ipynb").is_file():
    output_dir = cwd / "a7_part2"
elif cwd.name == "a7_part2" and (cwd / "Part2_RAG.ipynb").is_file():
    output_dir = cwd
else:
    raise FileNotFoundError("Launch from the repository root or a7_part2.")
assert len(rag_df) == 20
assert rag_df["quality_score"].between(1, 10).all()
output_path = output_dir / "part2_rag_results_revised.csv"
rag_df.to_csv(output_path, index=False)
print("Saved corrected RAG runs:", output_path)
print("Historical part2_rag_results.csv was not written.")

Saved corrected RAG runs: D:\INFO490\3_CozyDay\a7_part2\part2_rag_results_revised.csv
Historical part2_rag_results.csv was not written.


In [11]:
top1_correct = sum(retriever.retrieve(q["question"], top_k=1)[0]["id"] == q["doc_id"]
                   for q in questions)
print("Top-1 retrieval accuracy:", top1_correct, "/ 5")
display(rag_df.groupby("top_k").agg(
    runs=("question_id", "count"), answer_visible=("answer_visible_in_context", "sum"),
    answer_found=("answer_found", "sum"), mean_quality=("quality_score", "mean")))
print("Unique effective contexts per question (all K combined):")
display(rag_df.groupby("question_id")["actual_context"].nunique())

Top-1 retrieval accuracy: 5 / 5


,runs,answer_visible,answer_found,mean_quality
top_k,,,,
1,5,5,0,1.0
2,5,5,0,1.0
3,5,5,0,1.0
5,5,5,0,1.0


Unique effective contexts per question (all K combined):


question_id
Q1    1
Q2    1
Q3    1
Q4    1
Q5    1
Name: actual_context, dtype: int64

## Step 2.4: Required RAG analysis (corrected experiment)

### 1. Did providing retrieved information change the SLM's responses?
The corrected RAG responses differ from the authoritative matched-format baseline for four of five questions, while Q1 is unchanged across all K values. This comparison holds the question-ending space and generation settings fixed; the original question-only baseline is retained only as historical evidence.

### 2. Did RAG improve factual accuracy?
No: neither the five matched-format baseline responses nor the 20 corrected RAG responses provides the expected preference answer (0/5 and 0/20). All receive a revised quality proxy score of 1/10; these are rule-based assessments, not human ratings.

### 3. Did retrieving more documents always improve the responses?
No: Top-1, Top-2, Top-3, and Top-5 produce the same context and response for each question after suffix truncation. Thus this experiment does not isolate the effect of supplying more visible evidence, and identical scores do not establish that larger K is generally ineffective.

### 4. How did the SLM's context-length limitation affect the results?
Putting the highest-ranked document nearest the question preserves the expected answer in all 20 corrected inputs, compared with 5/20 originally. However, the 32-character suffix still removes the fact's subject and all earlier documents, so answer-span visibility must not be confused with visibility of the complete fact.

### 5. Why might RAG behave differently with a small character-level model compared with a larger pretrained language model?
This corpus-trained model has only 32 characters of context and was not trained to follow question-answer instructions, so retrieved text can be truncated or continued as unrelated training-style text. Larger pretrained language models often have longer context and richer language capabilities, although these results do not experimentally isolate model size or guarantee that a larger model would answer correctly.

## Authoritative matched-format baseline and quality evaluation

`part2_matched_baseline.csv` is the baseline Ashley requested for Parts 3 and 4: five unchanged questions, each followed by one space, the original checkpoint, seed 42, temperature 1.0, and 30 new characters. Its five responses and historical proxy scores are preserved byte-for-byte; the question-only baseline and original 25-run CSV are historical only.

### Revised scoring rubric

The new per-response table explicitly assesses factual accuracy, relevance, and support from the information actually visible to the model. These are conservative task-specific automatic proxies, not independent human evaluations or general semantic-entailment tests.

| Criterion | Rule on the 0–3 component scale |
|---|---|
| Factual accuracy | 3 for a complete accepted atomic answer (`answer` or `subject likes answer` after whitespace/punctuation normalization); 0 otherwise. |
| Relevance | 3 for an accepted answer; 1 for an exact subject-name mention without an accepted answer; 0 otherwise. |
| Retrieved-information support | 3 when an accepted answer matches an answer word present in the actual visible context; 0 otherwise. For no-RAG, this is explicitly N/A. |

The final 1–10 proxy equals `1 + 9 * mean(applicable components / 3)`. The baseline uses accuracy and relevance because it receives no retrieved evidence; RAG uses all three components. Report the components and answer accuracy alongside the overall proxy rather than interpreting the two applicable-component sets as a validated universal quality scale.

For these actual runs, every response is off-topic or fragmented, none supplies an accepted answer, and no RAG answer is supported by the retrieved preference evidence; all applicable components are zero, yielding 1/10. Historical 2/10 scores used a different word-count heuristic and remain unchanged in the original files.

The supplementary baseline below is regenerated to verify reproducibility against its existing CSV. The corrected comparison and all 25 component evaluations are exported separately at the end.

In [12]:
# Additional control: matched-format No-RAG baseline
# Preserve the original baseline and RAG results.

matched_baseline_results = []

for item in questions:
    matched_prompt = item["question"] + " "

    response, actual_context = rag_generate(
        matched_prompt,
        seed=SEED
    )

    matched_baseline_results.append({
        "question_id": item["id"],
        "question": item["question"],
        "expected_answer": item["answer"],
        "actual_context": actual_context,
        "response": response,
        "quality_score": quality_score(
            response,
            item["answer"]
        )
    })

matched_baseline_df = pd.DataFrame(matched_baseline_results)

display(matched_baseline_df)

print(
    "Matched-format baseline average:",
    matched_baseline_df["quality_score"].mean()
)

print(
    "Responses changed:",
    sum(
        matched_baseline_df["response"].tolist()[i]
        != baseline_df["response"].tolist()[i]
        for i in range(len(questions))
    ),
    "/ 5"
)

,question_id,question,expected_answer,actual_context,response,quality_score
0,Q1,what does mia like,apples,what does mia like,intelligence.\nstudents create,2
1,Q2,what does sam like,music,what does sam like,systems.\nlanguage models can r,2
2,Q3,what does tom like,swimming,what does tom like,s.\nattention helps models use,2
3,Q4,what does anna like,painting,what does anna like,intelligence.\nstudents create,2
4,Q5,what does lily like,reading,what does lily like,helps models use contextual in,2


Matched-format baseline average: 2.0
Responses changed: 5 / 5


In [13]:
from pathlib import Path
import pandas as pd

# Save supplementary matched-format baseline separately.
matched_export = matched_baseline_df.copy()
matched_export["method"] = "No-RAG matched-format"
matched_export["top_k"] = 0
matched_export["retrieved_ids"] = ""
matched_export["context_length"] = matched_export["actual_context"].str.len()
matched_export["answer_visible_in_context"] = False

columns = [
    "method",
    "question_id",
    "question",
    "expected_answer",
    "top_k",
    "retrieved_ids",
    "actual_context",
    "context_length",
    "response",
    "quality_score",
    "answer_visible_in_context"
]

matched_export = matched_export[columns]

# Locate the repository root from common Notebook working directories.
cwd = Path.cwd()
if (cwd / "a7_part2" / "Part2_RAG.ipynb").exists():
    output_dir = cwd / "a7_part2"
elif (cwd / "Part2_RAG.ipynb").exists() and cwd.name == "a7_part2":
    output_dir = cwd
else:
    raise FileNotFoundError("Cannot locate a7_part2 Notebook directory.")

output_path_matched = output_dir / "part2_matched_baseline.csv"
assert len(matched_export) == 5
assert matched_export["question_id"].is_unique
assert matched_export["quality_score"].between(1, 10).all()

if output_path_matched.exists():
    existing = pd.read_csv(output_path_matched, keep_default_na=False)
    pd.testing.assert_frame_equal(existing, matched_export, check_dtype=False)
    print("Existing authoritative baseline verified; file not overwritten.")
else:
    matched_export.to_csv(output_path_matched, index=False)

print("Saved:", output_path_matched)
print("Rows:", len(matched_export))

Existing authoritative baseline verified; file not overwritten.
Saved: D:\INFO490\3_CozyDay\a7_part2\part2_matched_baseline.csv
Rows: 5


In [14]:
# Current comparison and component evaluation; historical scores stay unchanged.
baseline_eval = matched_export.copy()
baseline_eval = baseline_eval.rename(columns={"quality_score": "historical_quality_score"})
new_eval = pd.DataFrame([
    evaluate_response(row.response, row.expected_answer, row.question)
    for row in baseline_eval.itertuples()
])
baseline_eval = pd.concat([baseline_eval.reset_index(drop=True), new_eval], axis=1)
revised_eval = rag_df.copy()
revised_eval["method"] = revised_eval["top_k"].map(lambda k: f"Top-{k} RAG revised")
quality_evaluation = pd.concat([baseline_eval, revised_eval], ignore_index=True)
assert len(quality_evaluation) == 25
assert quality_evaluation["quality_score"].between(1, 10).all()
assert rag_df["answer_visible_in_context"].all()
assert all(torch.equal(v.detach().cpu(), saved["model_state"][key])
           for key, v in model.state_dict().items())
quality_evaluation.to_csv(output_dir / "part2_quality_evaluation.csv", index=False)
print("Authoritative baseline: part2_matched_baseline.csv")
print("Corrected RAG results: part2_rag_results_revised.csv")
display(quality_evaluation[["method", "question_id", "top_k", "response",
    "factual_accuracy_0_3", "relevance_0_3", "retrieved_support_0_3", "quality_score"]])
display(quality_evaluation.groupby("method")["quality_score"].mean())
print("RAG response differences from authoritative baseline:")
control = matched_baseline_df.set_index("question_id")["response"]
display(rag_df.assign(changed=lambda df: df["response"] != df["question_id"].map(control))
        .groupby("top_k")["changed"].sum())

Authoritative baseline: part2_matched_baseline.csv
Corrected RAG results: part2_rag_results_revised.csv


,method,question_id,top_k,response,factual_accuracy_0_3,relevance_0_3,retrieved_support_0_3,quality_score
0,No-RAG matched-format,Q1,0,intelligence.\nstudents create,0,0,None,1.0
1,No-RAG matched-format,Q2,0,systems.\nlanguage models can r,0,0,None,1.0
2,No-RAG matched-format,Q3,0,s.\nattention helps models use,0,0,None,1.0
3,No-RAG matched-format,Q4,0,intelligence.\nstudents create,0,0,None,1.0
4,No-RAG matched-format,Q5,0,helps models use contextual in,0,0,None,1.0
5,Top-1 RAG revised,Q1,1,intelligence.\nstudents create,0,0,0,1.0
6,Top-2 RAG revised,Q1,2,intelligence.\nstudents create,0,0,0,1.0
7,Top-3 RAG revised,Q1,3,intelligence.\nstudents create,0,0,0,1.0
8,Top-5 RAG revised,Q1,5,intelligence.\nstudents create,0,0,0,1.0
9,Top-1 RAG revised,Q2,1,s useful for machine learning,0,0,0,1.0


method
No-RAG matched-format    1.0
Top-1 RAG revised        1.0
Top-2 RAG revised        1.0
Top-3 RAG revised        1.0
Top-5 RAG revised        1.0
Name: quality_score, dtype: float64

RAG response differences from authoritative baseline:


top_k
1    4
2    4
3    4
5    4
Name: changed, dtype: int64